# GoLLeM-v6-250M → zestaw SAE (TopK, strumień resztkowy po bloku)\n\nTrenuje **20 TopK-SAE** (po jednym na warstwę, `res_post_block`) dla [SlayerLab/GoLLeM-v6-250M](https://huggingface.co/SlayerLab/GoLLeM-v6-250M) — ten sam przepis co `res-v5` dla v5-128M:\n\n* `f = TopK(ReLU((x - b_dec) @ W_enc + b_enc))`, `x_hat = f_sparse @ W_dec + b_dec` (k=50, niewiązane macierze, wiersze dekodera renormowane do 1 po każdym kroku, `b_dec` inicjalizowane średnią resztą)\n* dane: **WikiText (EN) + Wikipedia PL** po połowie — v6 jest dwujęzyczny (2/3 PL), trening wyłącznie na angielskim zniekształciłby featury\n* rozmiar SAE: 960 → **7680** (8×), 655 kroków × batch 8 × 512 tokenów ≈ 2.68M tokenów — identycznie jak v5\n\n**Wyjście** (zgodne z `res-v5`, wchodzi prosto do stosu Neuronpedia): `res_v6_layer{L}_final.safetensors` (`W_enc` [960, 7680], `b_enc`, `W_dec` [7680, 960], `b_dec`, fp32), `sae_config.json`, `README.md` z SHA-256.\n\n**Wymagania:** Colab GPU (T4/L4), ~2–3 h na pełny przebieg. Model ładowany jest oficjalnym `modeling_gollem_v6.py` z repo modelu (inferencja-only, `strict=True`).\n\n**Licencje:** model CC-BY-SA-4.0 → wytrenowane SAE są bytem pochodnym — przy publikacji zachowaj atrybucję (SlayerLab/Fabryka AI) i tę samą licencję. Kod w komórkach możesz używać swobodnie (Apache-2.0, repo `neurony`).

## 1. Instalacja i pobranie modelu

In [ ]:
!pip install -q huggingface_hub datasets tokenizers safetensors

In [ ]:
import json, math, os, time, types\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom safetensors.torch import save_file\nfrom huggingface_hub import hf_hub_download\n\nassert torch.cuda.is_available(), 'Włącz runtime GPU: Runtime > Change runtime type > T4'\nDEVICE = 'cuda'\nMODEL_ID = 'SlayerLab/GoLLeM-v6-250M'\nMODEL_DIR = '/content/gollem-v6'\nOUT_DIR = '/content/gollem-v6-sae'\nos.makedirs(MODEL_DIR, exist_ok=True)\nos.makedirs(OUT_DIR, exist_ok=True)\n\nfor f in ('config.json', 'model.safetensors', 'tokenizer.json', 'modeling_gollem_v6.py'):\n    hf_hub_download(MODEL_ID, f, local_dir=MODEL_DIR)\n\nimport sys\nsys.path.insert(0, MODEL_DIR)\nfrom modeling_gollem_v6 import load_gollem_v6  # oficjalny kod z repo modelu\n\ncfg = types.SimpleNamespace(**json.load(open(f'{MODEL_DIR}/config.json')))\nmodel, tok = load_gollem_v6(MODEL_DIR, device=DEVICE)\nprint(f'model: {cfg.n_layer} warstw, d_model={cfg.n_embd}, vocab={cfg.vocab} | device: {next(model.parameters()).device}')

## 2. Parametry SAE i dane\n\nParametry są zgodne z przepisem `res-v5`; `DATA_MIX` ustawia proporcję EN/PL (v6 jest 2/3 polski — domyślnie 50/50 dla zrównoważonych featurów).

In [ ]:
D_MODEL = cfg.n_embd        # 960\nD_SAE = 8 * D_MODEL        # 7680 (8x — jak w res-v5)\nK = 50                     # TopK\nSEQ = 512                  # długość okna (jak w res-v5)\nBATCH = 8                  # okien na krok (jak w res-v5)\nSTEPS = 655                # 655 x 8 x 512 = 2.68M tokenów (jak w res-v5)\nLR = 1e-3                  # Adam (jak w res-v5)\nDATA_MIX = {'en': 0.5, 'pl': 0.5}\nSEED = 1337\ntorch.manual_seed(SEED)\nn_tokens_target = STEPS * BATCH * SEQ\nprint(f'cel: {n_tokens_target:,} tokenów | SAE: {D_MODEL} -> {D_SAE}, k={K}')

In [ ]:
from datasets import load_dataset\n\ndef collect(ids_fn, budget):\n    ids = []\n    for add in ids_fn():\n        ids.extend(add)\n        if len(ids) >= budget:\n            break\n    return ids[:budget]\n\ndef en_docs():\n    # WikiText-103 + WikiText-2 raw — ten sam korpus co dla res-v5\n    for name in ('wikitext-103-raw-v1', 'wikitext-2-raw-v1'):\n        for row in load_dataset('Salesforce/wikitext', name, split='train', streaming=True):\n            t = row['text']\n            if t.strip():\n                yield tok.encode(t).ids\n\ndef pl_docs():\n    # Wikipedia PL — dla dwujęzycznego v6\n    for row in load_dataset('wikimedia/wikipedia', '20231101.pl', split='train', streaming=True):\n        t = row.get('text', '')\n        if len(t) > 200:\n            yield tok.encode(t[:20000]).ids\n\nt0 = time.time()\nn_en = int(n_tokens_target * DATA_MIX['en'])\nn_pl = n_tokens_target - n_en\nids_en = collect(en_docs, n_en) if n_en else []\nids_pl = collect(pl_docs, n_pl) if n_pl else []\nprint(f'EN: {len(ids_en):,} tok | PL: {len(ids_pl):,} tok | {time.time() - t0:.0f}s')\n\n# mieszanie okien EN/PL (kolejność okien losowa, wewnątrz okna tekst spójny)\nwindows = []\nfor ids in (ids_en, ids_pl):\n    ids = ids[:(len(ids) // SEQ) * SEQ]\n    windows.extend(np.asarray(ids, dtype=np.int64).reshape(-1, SEQ))\nrng = np.random.default_rng(SEED)\nrng.shuffle(windows)\nwindows = torch.from_numpy(np.stack(windows))\nassert len(windows) >= STEPS * BATCH, f'za mało danych: {len(windows)} okien < {STEPS * BATCH}'\nwindows = windows[:STEPS * BATCH + 24]  # 24 okna na estymację b_dec\nprint('okna:', tuple(windows.shape))

## 3. TopKSAE i zbieranie `res_post_block`\n\nKlasa SAE jest zgodna z `topk_sae.py` z repo [`PiotrSty/gollem-v5-128m-sae-res-v5`](https://huggingface.co/PiotrSty/gollem-v5-128m-sae-res-v5). Strumień resztkowy pobieramy dokładnie jak w treningu v5: **po wyjściu każdego bloku** (po obu rezydualach).

In [ ]:
class TopKSAE(nn.Module):\n    # zgodne z topk_sae.py z repo res-v5 (PiotrSty)\n    def __init__(self, d_in, d_sae, k):\n        super().__init__()\n        self.d_in, self.d_sae, self.k = d_in, d_sae, k\n        self.W_enc = nn.Parameter(torch.empty(d_in, d_sae))\n        self.b_enc = nn.Parameter(torch.zeros(d_sae))\n        self.W_dec = nn.Parameter(torch.empty(d_sae, d_in))\n        self.b_dec = nn.Parameter(torch.zeros(d_in))\n        w = torch.randn(d_sae, d_in)\n        with torch.no_grad():\n            w /= w.norm(dim=1, keepdim=True)\n            self.W_dec.copy_(w)\n            self.W_enc.copy_(w.t() * 0.2)\n\n    @torch.no_grad()\n    def renorm_decoder(self):\n        self.W_dec.div_(self.W_dec.norm(dim=1, keepdim=True).clamp_min(1e-8))\n\n    def encode(self, x):\n        pre = F.relu((x - self.b_dec) @ self.W_enc + self.b_enc)\n        vals, idx = pre.topk(self.k, dim=-1)\n        return vals, idx\n\n    def decode(self, vals, idx):\n        f = torch.zeros(vals.size(0), self.d_sae, device=vals.device, dtype=vals.dtype)\n        f.scatter_(-1, idx, vals)\n        return f @ self.W_dec + self.b_dec\n\n    def forward(self, x):\n        vals, idx = self.encode(x)\n        return self.decode(vals, idx), vals, idx\n\n\n@torch.no_grad()\ndef resid_post(idx):\n    '''[n_layer, B*T, d_model] — strumień resztkowy po każdym bloku (res_post_block).'''\n    x = model.tok(idx.to(DEVICE))\n    v0, out = None, []\n    for b in model.blocks:\n        x, v0 = b(x, v0)\n        out.append(x)\n    return torch.stack([o.reshape(-1, D_MODEL) for o in out])

## 4. Inicjalizacja `b_dec` średnią resztą i trening\n\nJeden przebieg danych: forward modelu daje residua wszystkich 20 warstw, a każda warstwa robi własny krok Adama. (Dzięki temu model liczymy raz, a nie 20 razy.)

In [ ]:
saes = [TopKSAE(D_MODEL, D_SAE, K).to(DEVICE) for _ in range(cfg.n_layer)]\nopts = [torch.optim.Adam(s.parameters(), lr=LR) for s in saes]\n\n# b_dec = średni strumień resztkowy (jak w res-v5)\nt0 = time.time()\nmean = torch.zeros(cfg.n_layer, D_MODEL, device=DEVICE)\nfor i in range(0, 24, BATCH):\n    rs = resid_post(windows[i:i + BATCH])\n    mean += rs.mean(dim=1)\nmean /= 24 / BATCH\nfor l in range(cfg.n_layer):\n    saes[l].b_dec.data.copy_(mean[l])\nprint(f'b_dec zainicjalizowane w {time.time() - t0:.0f}s')

In [ ]:
t0 = time.time()\nfor step in range(STEPS):\n    batch = windows[24 + step * BATCH: 24 + (step + 1) * BATCH]\n    with torch.no_grad():\n        rs = resid_post(batch)                     # [20, 4096, 960]\n    for l in range(cfg.n_layer):\n        x = rs[l]\n        with torch.autocast('cuda', dtype=torch.float16):\n            x_hat, vals, idx = saes[l](x)\n            loss = F.mse_loss(x_hat, x)\n        opts[l].zero_grad(set_to_none=True)\n        loss.backward()\n        opts[l].step()\n        saes[l].renorm_decoder()\n    if (step + 1) % 50 == 0:\n        dt = time.time() - t0\n        print(f'step {step + 1}/{STEPS}  loss(last layer) {loss.item():.4f}  {dt:.0f}s  ETA {(STEPS - step - 1) * dt / (step + 1) / 60:.0f} min', flush=True)\nprint(f'trening: {(time.time() - t0) / 60:.1f} min')

## 5. Metryki: FVU i odsetek martwych featurów\n\nOdnośnik `res-v5`: FVU 0.07–0.24, dead frac ≈ 0%.

In [ ]:
@torch.no_grad()\ndef evaluate():\n    fvu_all, dead_all, hits = [], [], torch.zeros(cfg.n_layer, D_SAE, device=DEVICE)\n    n_tok = 0\n    for i in range(0, 24, BATCH):           # okna trzymane z boku (poza treningiem)\n        rs = resid_post(windows[STEPS * BATCH + i: STEPS * BATCH + i + BATCH])\n        n_tok += BATCH * SEQ\n        for l in range(cfg.n_layer):\n            x = rs[l]\n            x_hat, vals, idx = saes[l](x)\n            err = (x - x_hat).pow(2).sum()\n            tot = (x - x.mean(0, keepdim=True)).pow(2).sum()\n            fvu_all.append((err / tot.clamp_min(1e-12)).item())\n            hits.scatter_add_(1, idx, torch.ones_like(vals))\n    for l in range(cfg.n_layer):\n        dead_all.append((hits[l] == 0).float().mean().item())\n    return fvu_all, dead_all\n\nfvus, deads = evaluate()\nprint(f'{"layer":>5} {"FVU":>7} {"dead_frac":>9}')\nfor l in range(cfg.n_layer):\n    print(f'{l:>5} {fvus[l]:>7.4f} {deads[l]:>9.4f}')

## 6. Zapis artefaktów (jak `res-v5`) + SHA-256

In [ ]:
import hashlib, zipfile\n\nsha_rows = []\nfor l in range(cfg.n_layer):\n    s = saes[l]\n    path = f'{OUT_DIR}/res_v6_layer{l}_final.safetensors'\n    save_file({\n        'W_enc': s.W_enc.detach().float().cpu().contiguous(),   # [d_in, d_sae] — orientacja jak w res-v5\n        'b_enc': s.b_enc.detach().float().cpu().contiguous(),\n        'W_dec': s.W_dec.detach().float().cpu().contiguous(),  # [d_sae, d_in]\n        'b_dec': s.b_dec.detach().float().cpu().contiguous(),\n    }, path)\n    sha_rows.append((os.path.basename(path), hashlib.sha256(open(path, 'rb').read()).hexdigest()))\n\nsae_config = {\n    'model_id': 'gollem-v6-250m', 'hook': 'res_post_block', 'layers': cfg.n_layer,\n    'd_in': D_MODEL, 'd_sae': D_SAE, 'k': K, 'lr': LR,\n    'batch': BATCH, 'seq': SEQ, 'step': STEPS, 'tokens': STEPS * BATCH * SEQ,\n}\njson.dump(sae_config, open(f'{OUT_DIR}/sae_config.json', 'w'), indent=1)\n\nlines = [f'# GoLLeM-v6-250M SAE set - residual stream (res_v6)\\n',\n         f'TopK SAEs on the post-block residual stream of {MODEL_ID}, one per layer, {cfg.n_layer} layers.\\n',\n         f'd_in {D_MODEL} -> d_sae {D_SAE} (8x), k = {K}. Data: WikiText + Polish Wikipedia ({int(DATA_MIX["en"]*100)}/{int(DATA_MIX["pl"]*100)}).\\n',\n         f'Training: {STEPS} steps x batch {BATCH} x seq {SEQ} = {STEPS*BATCH*SEQ:,} tokens, Adam lr {LR}, decoder renorm per step, b_dec = mean residual.\\n',\n         '\\n## Metrics\\n', f'| layer | FVU | dead frac |\\n|---|---:|---:|\\n']\nlines += [f'| {l} | {fvus[l]:.4f} | {deads[l]:.4f} |\\n' for l in range(cfg.n_layer)]\nlines += ['\\n## Files\\n| file | sha256 |\\n|---|---|\\n']\nlines += [f'| {n} | `{h}` |\\n' for n, h in sha_rows]\nopen(f'{OUT_DIR}/README.md', 'w').writelines(lines)\n\nwith zipfile.ZipFile('/content/gollem-v6-sae.zip', 'w', zipfile.ZIP_DEFLATED) as z:\n    for f in os.listdir(OUT_DIR):\n        z.write(f'{OUT_DIR}/{f}', f)\nprint('gotowe:', sorted(os.listdir(OUT_DIR)))\nprint('paczka: /content/gollem-v6-sae.zip (File > Download)')

## 7. (Opcjonalnie) publikacja na Hugging Face\n\nOdkomentuj, jeśli chcesz opublikować zestaw jak `PiotrSty/gollem-v5-128m-sae-res-v5`. Pamiętaj o licencji modelu (CC-BY-SA-4.0) — opis w README paczki jest już przygotowany.

In [ ]:
# from huggingface_hub import login, HfApi\n# login()  # token z write\n# HfApi().upload_folder(folder_path=OUT_DIR, repo_id='TwojLogin/gollem-v6-250m-sae-res-v6', repo_type='model')

## 8. Dalsze kroki w stosie Neuronpedia (repo `neurony`)\n\n1. Zmień stałe w `make_sae_cache.py` na `D_MODEL=960, D_SAE=7680, K=50, N_LAYER=20` i wygeneruj cache circuit-tracera (self-test sprawdzi dokładność konwersji).\n2. Zarejestruj model i source set `res-v6` w webappie (przewodnik PDF, sekcje 3–5) i wgraj featury przez `upload_features.py` (najpierw `mine_activations.py` — analogicznie, z `modeling_gollem_v6.py` zamiast `hf_wrap`).\n3. Grafy atrybucji: `start_graph_gollem.py` z modelem v6 — ten sam schemat, nowy `TRANSCODER_SET`.\n\n*Szczegóły: `GoLLeM-Neuronpedia-przewodnik.pdf` oraz `github.com/PiotrStyla/neurony` (NOTICE — licencje i atrybucje).*